# Working with larger-than-memory data

Every chapter so far loaded data fully into memory. Real datasets don't always
fit. `polars`' **lazy** and **streaming** execution let you build a pipeline
that never materializes the whole thing at once.

```{note}
We demonstrate on the small [ETL dataset](../01c-etl/data-preparation.ipynb) for
practicality — but the *technique* is what scales, not the demo data. The same
code runs unchanged on a file far too big for RAM.
```

## The lazy API: build a plan, don't run it

`LazyCsvReader` *scans* rather than reads — it records what you want without
touching the data. Chaining `filter`/`select` builds a **query plan**; nothing
executes until `.collect()`.

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "csv", "streaming"] }
:dep ndarray = { version = "0.15" }
:dep incremental-rs = { version = "0.1.2" }
use polars::prelude::*;

let query: LazyFrame = LazyCsvReader::new("/book/data/customers.csv")
    .with_has_header(true)
    .finish()?
    .filter(col("churned").eq(lit(1)))
    .select([col("city"), col("income")]);
println!("query plan built — no data read yet");

## Query optimization: pushdown

Before running, polars *optimizes* the plan. `explain(true)` shows the optimized
version — note how the filter and the column selection are **pushed down** to the
CSV scan, so only the needed rows and columns are ever read from disk:

In [ ]:
println!("{}", query.clone().explain(true)?);

That pushdown is the whole game: on a 100 GB file, reading only two columns of
the matching rows is the difference between feasible and impossible.

## Streaming execution

`with_streaming(true)` runs the query in **chunks** that fit in memory, rather
than loading the full input. The result is identical; the memory profile isn't:

In [ ]:
{
    let result = query.clone().with_streaming(true).collect()?;
    println!("streamed result: {} rows x {} cols", result.height(), result.width());
    println!("{}", result);
}

## Out-of-core training: incremental fitting

Streaming fixed the *data* side. Training can be out-of-core too:
[`incremental-rs`](https://crates.io/crates/incremental-rs) provides `partial_fit`
estimators — SGD linear / logistic regression, Welford-online Gaussian Naive
Bayes, and mini-batch k-means — that update from **one batch at a time**, so the
full training set never has to sit in RAM. Here we fit a linear regression from a
stream of batches, recovering the true `y = 2·x1 − 3·x2 + 0.5` without ever
holding more than one batch:

In [ ]:
{
    use ndarray::{Array1, Array2};
    use incremental_rs::{IncrementalLinearRegression, IncrementalSupervisedEstimator, LearningRateSchedule};

    // An SGD linear model with a decaying learning-rate schedule.
    let mut model = IncrementalLinearRegression::new(
        LearningRateSchedule::InverseScaling { initial_rate: 0.2, decay: 0.001, power: 0.5 }, 0.0);

    // A stream of 300 batches of 16 rows, generated on the fly — a stand-in for
    // reading fixed-size chunks off disk. Only one batch is ever in memory.
    for b in 0..300u64 {
        let (mut xs, mut ys) = (Vec::with_capacity(32), Vec::with_capacity(16));
        for k in 0..16u64 {
            let s = b.wrapping_mul(2654435761).wrapping_add(k.wrapping_mul(40503));
            let x1 = ((s >> 8) & 0xff) as f64 / 255.0;
            let x2 = ((s >> 16) & 0xff) as f64 / 255.0;
            xs.push(x1); xs.push(x2);
            ys.push(2.0 * x1 - 3.0 * x2 + 0.5);
        }
        model.partial_fit(&Array2::from_shape_vec((16, 2), xs).unwrap(),
                          &Array1::from_vec(ys)).unwrap();
    }

    // The model learned the coefficients from the stream, never holding it whole.
    let test = Array2::from_shape_vec((2, 2), vec![1.0, 1.0, 2.0, 0.0]).unwrap();
    let p = model.predict(&test).unwrap();
    println!("fitted from the stream — predictions:");
    println!("  x=[1, 1] -> {:.3}   (true -0.5)", p[0]);
    println!("  x=[2, 0] -> {:.3}   (true  4.5)", p[1]);
}

## Where this stops

```{note}
Historically the modelling crates (`linfa`, `smartcore`) needed the whole
`ndarray` / `DenseMatrix` in memory, so "larger than memory" applied only to the
**ETL / feature-engineering** stage. That's no longer the full story:
[`incremental-rs`](https://crates.io/crates/incremental-rs) brings **out-of-core
training** to the algorithms that admit it — SGD linear / logistic regression,
online Naive Bayes, mini-batch k-means (above). Models with **no** incremental
form — decision trees, random forests, exact closed-form OLS — still need their
inputs in RAM. (`incremental-rs` also has an optional `polars-streaming` feature
that fits straight from a parquet stream; we drove `partial_fit` by hand here to
stay on this book's `polars` version.)
```

The practical pattern is still often the simplest: use lazy + streaming polars to
reduce a huge raw dataset down to the aggregated, filtered feature matrix that
*does* fit, then train on that — and now, when even that won't fit, reach for an
incremental estimator. It closes the loop on the
[ETL chapter](../01c-etl/data-preparation.ipynb), at scale.

Next: the [Capstone](../12-capstone/end-to-end-project.ipynb) — every chapter's
technique on one dataset, start to finish.